# Predictive Analytics Using Historical Data

**Objective:** Build a predictive model to forecast future sales trends using historical data.

### Key Features
- Clean and preprocess historical data
- Create time-based features
- Train a regression model for prediction
- Evaluate model accuracy
- Visualize historical trends and future predictions


## 1. Import Libraries

We use Pandas and NumPy for data handling, Matplotlib for visualization, and Scikit-learn for regression and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True


## 2. Load Historical Data

In [ ]:
df = pd.read_csv('historical_sales_data.csv', parse_dates=['Date'])
df.head()

## 3. Data Cleaning and Preprocessing

In [ ]:
print('Dataset shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())

# Remove duplicate rows if any
df = df.drop_duplicates().sort_values('Date').reset_index(drop=True)

# Time-based features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Time_Index'] = np.arange(len(df))

df.describe()

## 4. Explore Historical Trends

In [ ]:
plt.plot(df['Date'], df['Sales'], marker='o')
plt.title('Historical Monthly Sales')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.scatter(df['Marketing_Spend'], df['Sales'])
plt.title('Marketing Spend vs Sales')
plt.xlabel('Marketing Spend')
plt.ylabel('Sales')
plt.show()

## 5. Prepare Data for Regression

The last 8 months are kept as a test set so that the model is evaluated on future-like observations rather than random rows.

In [ ]:
features = ['Time_Index', 'Month', 'Marketing_Spend', 'Customers']
target = 'Sales'

train = df.iloc[:-8].copy()
test = df.iloc[-8:].copy()

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print('Training rows:', len(train))
print('Testing rows:', len(test))

## 6. Train the Predictive Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

test_predictions = model.predict(X_test)
test_results = test[['Date', 'Sales']].copy()
test_results['Predicted_Sales'] = test_predictions
test_results

## 7. Evaluate Model Accuracy

In [ ]:
mae = mean_absolute_error(y_test, test_predictions)
rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
r2 = r2_score(y_test, test_predictions)

print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'R²   : {r2:.3f}')

### Interpretation
- **MAE** shows the average absolute prediction error.
- **RMSE** penalizes larger prediction errors more strongly.
- **R²** indicates how much of the variation in sales is explained by the model.


## 8. Compare Actual and Predicted Sales

In [ ]:
plt.plot(test_results['Date'], test_results['Sales'], marker='o', label='Actual Sales')
plt.plot(test_results['Date'], test_results['Predicted_Sales'], marker='o', label='Predicted Sales')
plt.title('Actual vs Predicted Sales')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Forecast the Next 6 Months

In [ ]:
# Refit on all available historical observations
final_model = LinearRegression()
final_model.fit(df[features], df[target])

future_dates = pd.date_range(df['Date'].max() + pd.offsets.MonthBegin(1), periods=6, freq='MS')
future = pd.DataFrame({'Date': future_dates})
future['Time_Index'] = np.arange(len(df), len(df) + 6)
future['Month'] = future['Date'].dt.month

# Use recent averages as reasonable future input assumptions
future['Marketing_Spend'] = df['Marketing_Spend'].tail(6).mean()
future['Customers'] = df['Customers'].tail(6).mean()
future['Forecast_Sales'] = final_model.predict(future[features])

future[['Date', 'Forecast_Sales']]

In [ ]:
plt.plot(df['Date'], df['Sales'], marker='o', label='Historical Sales')
plt.plot(future['Date'], future['Forecast_Sales'], marker='o', label='6-Month Forecast')
plt.axvline(df['Date'].max(), linestyle='--', label='Forecast Start')
plt.title('Historical Sales and Future Forecast')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. Conclusion

The predictive analytics workflow cleaned historical data, created useful time-based features, trained a regression model, evaluated prediction accuracy, and generated a six-month sales forecast. The results demonstrate how historical trends and business variables such as marketing spend and customer count can support data-driven forecasting.

**Expected outcome achieved:** predictive modeling, trend analysis, model evaluation, visualization, and future forecasting.